In [31]:
from pathlib import Path
import sys

# 현재 노트북 위치에서 "Model" 폴더를 찾아 sys.path에 추가
cur = Path.cwd().resolve()

# 케이스 1) 노트북이 CVProject/Model 안에 있으면 model_dir = cur
# 케이스 2) 노트북이 CVProject/Model/notebooks 안에 있으면 model_dir = cur.parent
if (cur / "src").exists():
    model_dir = cur
elif (cur.parent / "src").exists():
    model_dir = cur.parent
else:
    raise FileNotFoundError("cannot find Model/src. please open notebook under CVProject/Model or CVProject/Model/notebooks")

sys.path.insert(0, str(model_dir))

print("model_dir:", model_dir)
print("sys.path[0]:", sys.path[0])

model_dir: /root/CVProject/Model
sys.path[0]: /root/CVProject/Model


### 캐시 초기화

In [32]:
# ===== 강제 모듈 초기화 (캐시 제거) =====
import sys, importlib

purge = [
    "src",
    "src.setup",
    "src.transforms",
    "src.dataset",
    "src.models",
    "src.trainer",
    "src.metrics",
    "src.dirty_holdout",
]

for m in purge:
    if m in sys.modules:
        del sys.modules[m]

# src 패키지를 새로 import
import src
import src.setup as setup
import src.transforms as T

print("✅ purged & reimported")
print("setup file:", setup.__file__)
print("transforms file:", T.__file__)


✅ purged & reimported
setup file: /root/CVProject/Model/src/setup.py
transforms file: /root/CVProject/Model/src/transforms.py


### 불러오기

In [33]:
import os
import numpy as np
import pandas as pd
import random
import torch
from tqdm import tqdm

from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedShuffleSplit              # 새로운 Hold-out 을 위한 새로운 선언문
from sklearn.metrics import f1_score

import wandb
from datetime import datetime

import src.setup as setup
from src.setup import cfg, seed_everything, load_train_df, train_dir, ckpt_dir
from src.dataset import doc_dataset
from src.transforms import get_train_transforms, get_valid_transforms
from src.trainer import predict_proba_with_targets

# 진단 Code
import src.transforms as T
import inspect
import importlib

# TTA
import math
import torch.nn.functional as F
from torch.utils.data import Sampler

# Aggressive vs Safe 
from pathlib import Path
from src.models import doc_classifier
from src.trainer import valid_one_epoch
from src.metrics import macro_f1

# Repeat Dataset
from torch.utils.data import Dataset

# train_on_epoch 호출부
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

print("TRANSFORMS FILE PATH:", T.__file__)
print("Has ShiftScaleRotate?", "ShiftScaleRotate" in inspect.getsource(T.get_train_transforms))

# 강제로 reload 해서 최신 파일 반영
importlib.reload(T)

print("AFTER RELOAD | Has ShiftScaleRotate?",
      "ShiftScaleRotate" in inspect.getsource(T.get_train_transforms))

from src.models import doc_classifier
from src.trainer import train_one_epoch, valid_one_epoch
from src.metrics import macro_f1
from src.losses import SoftTargetCrossEntropy

print("imports ok ✅")
print("cuda:", torch.cuda.is_available())


TRANSFORMS FILE PATH: /root/CVProject/Model/src/transforms.py
Has ShiftScaleRotate? False
AFTER RELOAD | Has ShiftScaleRotate? False
imports ok ✅
cuda: True


### Bucket

In [34]:
def make_bucket_resolutions(base_size=640, min_dim=320, max_dim=768, step=64):
    """
    base_size^2 면적을 대략 유지하면서 다양한 (W,H) 버킷 생성
    """
    resolutions = set()
    target_area = base_size * base_size

    w = min_dim
    while w <= max_dim:
        h = max(1, int(target_area / w))
        w_snap = max(step, (w // step) * step)
        h_snap = max(step, (h // step) * step)

        if min_dim <= h_snap <= max_dim:
            resolutions.add((w_snap, h_snap))
            resolutions.add((h_snap, w_snap))
        w += step

    return sorted(list(resolutions))

In [35]:
class BucketBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, bucket_resolutions, shuffle=True, drop_last=False):
        self.dataset = dataset
        self.batch_size = batch_size
        self.bucket_resolutions = bucket_resolutions
        self.shuffle = shuffle
        self.drop_last = drop_last

        self.buckets = {res: [] for res in bucket_resolutions}
        self._assign_buckets()

    def _assign_buckets(self):
        print("Grouping images into buckets...")
        for idx in range(len(self.dataset)):
            w, h = self.dataset.get_image_size(idx)
            ar = float(w) / float(h) if h != 0 else 1.0
            best_res = min(self.bucket_resolutions, key=lambda res: abs((res[0] / res[1]) - ar))
            self.buckets[best_res].append(idx)
            self.dataset.set_target_resolution(idx, best_res)

    def __iter__(self):
        batches = []
        for res, indices in self.buckets.items():
            if self.shuffle:
                random.shuffle(indices)
            for i in range(0, len(indices), self.batch_size):
                batch = indices[i:i + self.batch_size]
                if self.drop_last and len(batch) < self.batch_size:
                    continue
                batches.append(batch)

        if self.shuffle:
            random.shuffle(batches)

        for batch in batches:
            yield batch

    def __len__(self):
        count = 0
        for indices in self.buckets.values():
            if self.drop_last:
                count += len(indices) // self.batch_size
            else:
                count += math.ceil(len(indices) / self.batch_size)
        return count

### Repeat Dataset

In [36]:
class RepeatDataset(Dataset):
    """RepeatAugment: dataset을 N배 길이로 보이게 해서 aug 노출을 증가"""
    def __init__(self, dataset, repeats=4):
        self.dataset = dataset
        self.repeats = int(repeats)

    def __len__(self):
        return len(self.dataset) * self.repeats

    def __getitem__(self, idx):
        return self.dataset[idx % len(self.dataset)]

    # ✅ Bucket sampler가 쓰는 메서드 delegate
    def get_image_size(self, idx):
        return self.dataset.get_image_size(idx % len(self.dataset))

    def set_target_resolution(self, idx, res):
        return self.dataset.set_target_resolution(idx % len(self.dataset), res)

### train_df Load

In [37]:
seed_everything(cfg["seed"])

train_df = load_train_df()
num_classes = train_df["target"].nunique()

print("train_df:", train_df.shape)
print("num_classes:", num_classes)
train_df["target"].value_counts().head()


train_df: (1570, 2)
num_classes: 17


target
16    100
10    100
4     100
5     100
15    100
Name: count, dtype: int64

### fold split + OOF Bufer

In [38]:
# ✅ bucket 기준 해상도 (setup.py의 bucket_base_size 사용)
base_size = cfg.get("bucket_base_size", cfg["img_size"])
print(f"[INFO] bucket base_size = {base_size}")

skf = StratifiedKFold(
    n_splits=cfg["num_folds"],
    shuffle=True,
    random_state=cfg["seed"]
)

oof_probs = np.zeros((len(train_df), num_classes), dtype=np.float32)
oof_true = train_df["target"].values

# ===== load subset soft targets made by 04_analysis_errors.ipynb =====
oof_dir = Path(cfg["paths"]["oof_dir"])
soft_idx_path  = oof_dir / "soft_targets_idx.npy"
soft_prob_path = oof_dir / "soft_targets_prob.npy"

use_soft_subset = soft_idx_path.exists() and soft_prob_path.exists()
print("use_soft_subset:", use_soft_subset)

soft_idx = None
soft_prob = None
soft_map = None  # orig_idx -> row position

if use_soft_subset:
    soft_idx = np.load(soft_idx_path).astype(np.int64)          # (Ns,)
    soft_prob = np.load(soft_prob_path).astype(np.float32)      # (Ns, C)
    assert soft_prob.shape[1] == num_classes

    # orig_idx -> position in soft_prob
    soft_map = {int(soft_idx[i]): i for i in range(len(soft_idx))}
    print("loaded soft subset:", len(soft_idx), "samples")


# =========================
# Dirty holdout split (Test CSV 기반)
# 기존 LB-proxy holdout 대체
# =========================
from src.dirty_holdout import load_test_thresholds, compute_train_metrics, build_dirty_holdout

use_dirty = bool(cfg.get("use_dirty_holdout", True))

if use_dirty:
    test_csv = cfg["paths"].get("test_dirty_metrics", None)
    assert test_csv is not None, "cfg['paths']['test_dirty_metrics'] is missing"
    assert Path(test_csv).exists(), f"missing test_dirty_metrics.csv: {test_csv}"

    th_cfg = cfg.get("dirty_thresholds", {})
    thresholds = load_test_thresholds(
        test_dirty_csv=test_csv,
        p_blur=float(th_cfg.get("p_blur", 0.10)),
        p_std=float(th_cfg.get("p_std", 0.10)),
        p_mean_lo=float(th_cfg.get("p_mean_lo", 0.10)),
        p_mean_hi=float(th_cfg.get("p_mean_hi", 0.90)),
        p_border=float(th_cfg.get("p_border", 0.90)),
    )

    # train metric cache (oof_dir 아래에 저장 추천)
    train_metrics_cache = Path(cfg["paths"]["oof_dir"]) / "train_dirty_metrics.csv"

    train_metrics_df = compute_train_metrics(
        train_df=train_df.reset_index(drop=True),
        train_dir=train_dir,
        id_col="ID",
        cache_csv=train_metrics_cache,
        border_ratio=0.05,
    )

    dev_df, lb_df, dev_idx, lb_idx = build_dirty_holdout(
        train_df=train_df.reset_index(drop=True),
        train_metrics_df=train_metrics_df,
        thresholds=thresholds,
        holdout_ratio=float(cfg.get("dirty_holdout_ratio", 0.2)),
        seed=int(cfg.get("dirty_holdout_seed", cfg["seed"])),
        id_col="ID",
        label_col="target",
    )

    # dev_idx/lb_idx는 "reset_index(drop=True) 기준 index"로 맞추기
    # (위에서 train_df를 reset_index(drop=True)로 넣었으니 그냥 range index로 간주)
    dev_idx = np.array(dev_idx, dtype=np.int64)
    lb_idx  = np.array(lb_idx, dtype=np.int64)

    print("HOLDOUT MODE | use_dirty_holdout:", use_dirty)
    print("test_dirty_metrics path:", test_csv)
    print("lb_df size:", lb_df.shape)
    print("lb_df target dist top5:\n", lb_df["target"].value_counts().head())
    print("dirty_holdout dirty_ratio (recalc):", float((train_metrics_df["ID"].isin(lb_df["ID"])).mean()))


else:
    # fallback: 기존 랜덤 stratified holdout
    sss = StratifiedShuffleSplit(
        n_splits=1,
        test_size=cfg.get("lb_holdout_ratio", 0.2),
        random_state=cfg.get("lb_holdout_seed", cfg["seed"])
    )
    dev_idx, lb_idx = next(sss.split(train_df, train_df["target"]))

    dev_df = train_df.iloc[dev_idx].reset_index(drop=True)
    lb_df  = train_df.iloc[lb_idx].reset_index(drop=True)

    dev_idx = np.array(dev_idx)
    lb_idx  = np.array(lb_idx)

    print(f"✅ fallback LB-holdout | dev: {len(dev_df)} | lb_holdout: {len(lb_df)}")


# LB holdout loader (eval only)
if lb_df is not None:
    lb_ds = doc_dataset(
        lb_df,
        train_dir,
        transforms=T.get_valid_transforms_bucket(),  # ✅ 크기 변경 없는 valid bucket transform
        is_test=False,
        scan_sizes=False,
        base_size=base_size,                          # ✅ 고정 해상도 letterbox
        use_letterbox=True,
        pad_value=(255,255,255),
    )
    lb_loader = DataLoader(
        lb_ds,
        batch_size=cfg["batch_size"],
        shuffle=False,
        num_workers=cfg["num_workers"],
        pin_memory=True
    )
else:
    lb_loader = None

# ===== store fold-wise lb_holdout probabilities for ensemble optimization =====
lb_probs_by_fold = {}   # fold -> (N_lb, C)
lb_true_ref = None      # lb labels (same for all folds)


[INFO] bucket base_size = 640
use_soft_subset: True
loaded soft subset: 33 samples


[DirtyHoldout] dev=1256 | dirty_holdout=314 | dirty_ratio_in_holdout=0.462
[DirtyHoldout] thresholds: {'blur_p': 161.82611313532828, 'std_p': 23.37310031546333, 'mean_lo_p': 105.53522025770674, 'mean_hi_p': 213.68743454297532, 'border_p': 0.5365188956873799}
HOLDOUT MODE | use_dirty_holdout: True
test_dirty_metrics path: /root/CVProject/data/test_dirty_metrics.csv
lb_df size: (314, 2)
lb_df target dist top5:
 target
10    20
4     20
16    20
5     20
15    20
Name: count, dtype: int64
dirty_holdout dirty_ratio (recalc): 0.2


### LB 에서도 TTA 를 줘서 좋을지 안좋을지 로컬에서도 판단해보기

In [39]:
def rot90_batch(x: torch.Tensor, k: int) -> torch.Tensor:
    """x: (B,C,H,W), k=0/1/2/3 -> 0/90/180/270 CCW"""
    if k % 4 == 0:
        return x
    return torch.rot90(x, k=k, dims=(-2, -1))

In [40]:
def rotate_affine_batch(x: torch.Tensor, angle_deg: float, mode: str = "bilinear") -> torch.Tensor:
    """
    Small-angle rotation using affine_grid + grid_sample (pure torch).
    padding_mode='border' to avoid black corners.
    """
    if abs(angle_deg) < 1e-6:
        return x

    B, C, H, W = x.shape
    theta = angle_deg * math.pi / 180.0
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)

    A = x.new_zeros((B, 2, 3))
    A[:, 0, 0] = cos_t
    A[:, 0, 1] = -sin_t
    A[:, 1, 0] = sin_t
    A[:, 1, 1] = cos_t

    grid = F.affine_grid(A, size=x.size(), align_corners=False)
    x_rot = F.grid_sample(
        x, grid,
        mode=mode,
        padding_mode="border",
        align_corners=False
    )
    return x_rot

In [41]:
@torch.no_grad()
def valid_one_epoch_adaptive_tta(model, loader, use_amp: bool, small_angles=(-15.0, 0.0, 15.0)):
    """
    lb_holdout 평가용 TTA:
    Stage1) 0/90/180/270 중 샘플별 confidence 최대 회전 선택
    Stage2) 그 회전에서만 소각도(-15/0/+15) 적용 -> logits 평균 -> softmax
    return: targets(np), preds(np)
    """
    model.eval()
    preds_all, targets_all = [], []

    for x, y in tqdm(loader, leave=False):
        x = x.cuda(non_blocking=True)

        # ---- Stage 1: big rotations
        logits_list = []
        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            for k in (0, 1, 2, 3):
                xr = rot90_batch(x, k)
                logits_list.append(model(xr))  # (B, K)

        logits_stack = torch.stack(logits_list, dim=0)          # (4, B, K)
        probs_stack  = torch.softmax(logits_stack, dim=-1)      # (4, B, K)
        conf = probs_stack.max(dim=-1).values                   # (4, B)
        best_k = conf.argmax(dim=0)                             # (B,)

        # ---- Stage 2: small angles only on chosen big rotation
        B = x.size(0)
        logits_sum = None

        for ang in small_angles:
            x_base = x.new_empty(x.shape)
            for k in (0, 1, 2, 3):
                idx = (best_k == k).nonzero(as_tuple=True)[0]
                if idx.numel() == 0:
                    continue
                x_base[idx] = rot90_batch(x[idx], k)

            x_aug = rotate_affine_batch(x_base, float(ang))

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                logits = model(x_aug)

            logits_sum = logits if logits_sum is None else (logits_sum + logits)

        logits_avg = logits_sum / float(len(small_angles))
        preds = logits_avg.argmax(dim=1).detach().cpu().numpy()

        preds_all.append(preds)
        targets_all.append(y.numpy())

    preds_all = np.concatenate(preds_all, axis=0)
    targets_all = np.concatenate(targets_all, axis=0)
    return targets_all, preds_all

### Train 돌기전 Debug 체크

In [42]:
def debug_check_loader_shapes(train_loader, valid_loader, lb_loader=None, base_size=None):
    # train: bucket이라 H,W가 다양할 수 있음
    x, y = next(iter(train_loader))
    print(f"[DEBUG] train batch: {tuple(x.shape)} | dtype={x.dtype} | min/max={x.min().item():.3f}/{x.max().item():.3f}")

    # valid: 고정 letterbox면 base_size x base_size
    x, y = next(iter(valid_loader))
    print(f"[DEBUG] valid batch: {tuple(x.shape)}")

    if base_size is not None:
        if x.shape[-2:] != (base_size, base_size):
            print(f"[WARN] valid size mismatch! expected {(base_size, base_size)} but got {tuple(x.shape[-2:])}")

    # lb도 있으면 같이 확인
    if lb_loader is not None:
        x, y = next(iter(lb_loader))
        print(f"[DEBUG] lb batch   : {tuple(x.shape)}")


### Train Loop

In [43]:
# WanDB 에 실험 Tag 붙이기
exp_tag = datetime.now().strftime("%m%d_%H%M%S")  # 예: 0202_213501
base_name = cfg["wandb"]["run_name"]

for fold, (tr_idx, va_idx) in enumerate(skf.split(dev_df, dev_df["target"])):
    print(f"\n========== fold {fold} ==========")

    # wandb run: fold마다 하나씩 쌓임
    if cfg["wandb"]["enabled"]:
        wandb.init(
                project=cfg["wandb"]["project"],
                entity=cfg["wandb"]["entity"],
                name=f"{base_name}_{exp_tag}_fold{fold}",
                group=f"{base_name}_{exp_tag}",          # fold run들을 한 그룹으로 묶음
                job_type="train_kfold",
                config=cfg
        )

    tr_df = dev_df.iloc[tr_idx].reset_index(drop=True)
    va_df = dev_df.iloc[va_idx].reset_index(drop=True)

    # ===== build train soft target matrix (N_tr, C) =====
    soft_targets_tr = None

    if use_soft_subset:
        C = num_classes
        Ntr = len(tr_df)

        # (1) 기본 one-hot (모든 샘플을 벡터로 통일 -> DataLoader collate 절대 안 깨짐)
        soft_targets_tr = np.zeros((Ntr, C), dtype=np.float32)
        y_tr = tr_df["target"].values.astype(np.int64)
        soft_targets_tr[np.arange(Ntr), y_tr] = 1.0

        # (2) suspects(원본 train_df index 기준)만 soft_prob로 교체
        # tr_idx는 dev_df 기준 index -> 원본 train_df index는 dev_idx[tr_idx]
        orig_tr_idx = dev_idx[tr_idx]  # length == Ntr

        hit = 0
        for local_i, orig_i in enumerate(orig_tr_idx):
            j = soft_map.get(int(orig_i), None)
            if j is not None:
                soft_targets_tr[local_i] = soft_prob[j]  # (C,)
                hit += 1

        print(f"[fold {fold}] soft subset hits in train: {hit}/{Ntr}")


    # ✅ Bucketing용 버킷 해상도 리스트
    bucket_res_list = make_bucket_resolutions(
        base_size=base_size,
        min_dim=384,
        max_dim=896,
        step=64
    )

    # ✅ Train Dataset (Bucket + Letterbox)
    train_ds = doc_dataset(
        tr_df,
        train_dir,
        transforms=T.get_train_transforms_bucket(),
        is_test=False,
        scan_sizes=True,
        base_size=base_size,
        use_letterbox=True,
        pad_value=(255,255,255),
        soft_targets=soft_targets_tr,
    )

    # ✅ RepeatDataset 적용 (실험: 2~4 추천)
    # ✅ RepeatDataset repeats를 cfg로 제어 (빠른 실험: 1~2 권장)
    repeats = int(cfg.get("repeat_repeats", 1))

    if repeats > 1:
        train_ds_rep = RepeatDataset(train_ds, repeats=repeats)
    else:
        train_ds_rep = train_ds

    print(f"[INFO] RepeatDataset repeats={repeats} | base_len={len(train_ds)} | rep_len={len(train_ds_rep)}")


    # ✅ Bucket Sampler
    train_sampler = BucketBatchSampler(
        train_ds_rep,                                 # ✅ sampler는 반복 dataset에 붙임
        batch_size=cfg["batch_size"],
        bucket_resolutions=bucket_res_list,
        shuffle=True,
        drop_last=True
    )

    # ✅ Train Loader (batch_sampler 사용 → shuffle 인자 쓰면 안 됨)
    train_loader = DataLoader(
        train_ds_rep,                                 # ✅ loader도 반복 dataset
        batch_sampler=train_sampler,
        num_workers=cfg["num_workers"],
        pin_memory=True
    )

    print(f"[INFO] RepeatDataset repeats={repeats} | base_len={len(train_ds)} | rep_len={len(train_ds_rep)}")

    # ✅ Valid Dataset (고정 해상도 letterbox: BASE_SIZE x BASE_SIZE)
    valid_ds = doc_dataset(
        va_df,
        train_dir,
        transforms=T.get_valid_transforms_bucket(),  # ✅ bucket 전용 valid transforms (크기 조절 없음)
        is_test=False,
        scan_sizes=False,                             # ✅ valid은 스캔 불필요
        base_size=base_size,                          # ✅ 여기서 BASE_SIZE = 고정 해상도
        use_letterbox=True,
        pad_value=(255,255,255),
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=cfg["batch_size"],
        shuffle=False,
        num_workers=cfg["num_workers"],
        pin_memory=True
    )

    # Debug Check
    debug_check_loader_shapes(train_loader, valid_loader, lb_loader=lb_loader, base_size=base_size)

    model = doc_classifier(
        model_name=cfg["model_name"],
        num_classes=num_classes,
        pretrained=cfg["pretrained"],
        dropout=cfg["dropout"]
    ).cuda()

    criterion = SoftTargetCrossEntropy()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"]
    )

    scaler = torch.amp.GradScaler("cuda", enabled=cfg["use_amp"])

    # ✅ Scheduler (CosineAnnealingWarmRestarts)
    sch_cfg = cfg.get("scheduler", {})
    if sch_cfg.get("enabled", False) and sch_cfg.get("type", "") == "cosine_warm_restarts":
        scheduler = CosineAnnealingWarmRestarts(
            optimizer,
            T_0=int(sch_cfg.get("T_0", 10)),
            T_mult=int(sch_cfg.get("T_mult", 1)),
            eta_min=float(cfg.get("min_lr", 1e-6)),
        )
    else:
        scheduler = None

    best_val_f1 = -1.0
    best_lb_f1  = -1.0

    # ===== Early Stopping state =====
    es_cfg = cfg.get("early_stopping", {"enabled": False})
    es_enabled = bool(es_cfg.get("enabled", False))
    es_monitor = es_cfg.get("monitor", "lb")  # "lb" or "val"
    es_patience = int(es_cfg.get("patience", 5))
    es_min_delta = float(es_cfg.get("min_delta", 0.0))
    es_warmup = int(es_cfg.get("warmup", 0))

    best_mon = -1.0  # monitor best (lb or val)
    bad_epochs = 0

    best_val_path = Path(ckpt_dir) / f"best_fold{fold}_val.pth"
    best_lb_path  = Path(ckpt_dir) / f"best_fold{fold}_lb.pth"

    for epoch in range(cfg["epochs"]):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            scaler,
            use_amp=cfg["use_amp"],
            grad_clip_max_norm=float(cfg.get("grad_clip_max_norm", 0.0)),  # ✅ 추가
            scheduler=scheduler,    # ✅ 추가
            epoch=epoch,            # ✅ 추가
        )

        y_true, y_pred, y_prob = valid_one_epoch(model, valid_loader, use_amp=cfg["use_amp"])
        val_f1 = macro_f1(y_true, y_pred)

        # LB-proxy holdout eval
        lb_f1 = None
        if lb_loader is not None:
            lb_true, lb_pred, _ = valid_one_epoch(model, lb_loader, use_amp=cfg["use_amp"])
            lb_f1 = macro_f1(lb_true, lb_pred)

        if cfg["wandb"]["enabled"]:
            wandb.log({
                "fold": fold,
                "epoch": epoch,
                "train_loss": train_loss,
                "val_macro_f1": val_f1,
                "lb_holdout_macro_f1": lb_f1 if lb_f1 is not None else np.nan,

            })

        if lb_f1 is not None:
            print(f"[fold {fold}] epoch {epoch:02d} | loss {train_loss:.4f} | val_f1 {val_f1:.4f} | lb_f1 {lb_f1:.4f}")
        else:
            print(f"[fold {fold}] epoch {epoch:02d} | loss {train_loss:.4f} | val_f1 {val_f1:.4f}")
        
        # ===== Early stopping check =====
        if es_enabled:
            # monitor score 선택
            if es_monitor == "lb" and (lb_f1 is not None):
                mon = lb_f1
            else:
                mon = val_f1

            # warmup 동안은 체크하지 않음
            if epoch >= es_warmup:
                improved = mon > (best_mon + es_min_delta)
                if improved:
                    best_mon = mon
                    bad_epochs = 0
                else:
                    bad_epochs += 1

                if bad_epochs >= es_patience:
                    print(f"[fold {fold}] EarlyStopping triggered at epoch {epoch:02d} | best_{es_monitor}: {best_mon:.4f}")
                    break


        # choose best metric: prefer lb_f1 if available, otherwise val_f1
        score = lb_f1 if lb_f1 is not None else val_f1

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_val_path)

        if lb_f1 is not None and lb_f1 > best_lb_f1:
            best_lb_f1 = lb_f1
            torch.save(model.state_dict(), best_lb_path)

    print(f"[fold {fold}] best val_f1: {best_val_f1:.4f} | saved: {best_val_path}")
    print(f"[fold {fold}] best lb_f1 : {best_lb_f1:.4f} | saved: {best_lb_path}")

    # ===== LB-holdout에서 TTA 적용/미적용 비교 (제출 전 게이트) =====
    if lb_loader is not None:
        # best_lb ckpt 로드 (제출에 쓰는 기준과 동일)
        model.load_state_dict(torch.load(best_lb_path, map_location="cuda"))

        # (A) lb_holdout probs 저장 (앙상블 최적화용)
        lb_prob, lb_true = predict_proba_with_targets(model, lb_loader, use_amp=cfg["use_amp"])
        lb_probs_by_fold[fold] = lb_prob

        if lb_true_ref is None:
            lb_true_ref = lb_true
        else:
            # safety: 라벨 순서 동일해야 함
            assert np.array_equal(lb_true_ref, lb_true)


        # (1) 미적용: 기존 valid_one_epoch로 재평가 (저장된 best_lb_f1과 같은 조건 확인)
        lb_true_ntta, lb_pred_ntta, _ = valid_one_epoch(model, lb_loader, use_amp=cfg["use_amp"])
        lb_f1_ntta = macro_f1(lb_true_ntta, lb_pred_ntta)

        # (2) TTA 적용 평가
        lb_true_tta, lb_pred_tta = valid_one_epoch_adaptive_tta(
            model,
            lb_loader,
            use_amp=cfg["use_amp"],
            small_angles=(-15.0, 0.0, 15.0),
        )
        lb_f1_tta = macro_f1(lb_true_tta, lb_pred_tta)

        print(f"[fold {fold}] LB eval (best_lb ckpt) | no-TTA: {lb_f1_ntta:.4f} | TTA: {lb_f1_tta:.4f} | delta: {lb_f1_tta - lb_f1_ntta:+.4f}")

        if cfg["wandb"]["enabled"]:
            wandb.log({
                "lb_eval_no_tta": lb_f1_ntta,
                "lb_eval_tta": lb_f1_tta,
                "lb_eval_tta_delta": lb_f1_tta - lb_f1_ntta,
            })
    
    # ===== save lb probs for ensemble optimization =====
    lb_save_path = Path(cfg["paths"]["oof_dir"]) / "lb_probs_by_fold.npy"
    np.save(lb_save_path, lb_probs_by_fold, allow_pickle=True)
    print("saved:", lb_save_path)

    lb_true_path = Path(cfg["paths"]["oof_dir"]) / "lb_true.npy"
    np.save(lb_true_path, lb_true_ref)
    print("saved:", lb_true_path)

    # best ckpt로 oof 채우기
    oof_ckpt_path = best_val_path  # 기본: val 기준
    # oof_ckpt_path = best_lb_path  # 대안: lb_holdout 기준

    model.load_state_dict(torch.load(oof_ckpt_path, map_location="cuda"))
    _, _, prob = valid_one_epoch(model, valid_loader, use_amp=cfg["use_amp"])
    orig_va_idx = dev_idx[va_idx]
    oof_probs[orig_va_idx] = prob

    if cfg["wandb"]["enabled"]:
        wandb.finish()



========== fold 0 ==========
[fold 0] soft subset hits in train: 16/837


[INFO] RepeatDataset repeats=1 | base_len=837 | rep_len=837
Grouping images into buckets...
[INFO] RepeatDataset repeats=1 | base_len=837 | rep_len=837
[DEBUG] train batch: (8, 3, 512, 768) | dtype=torch.float32 | min/max=-2.118/2.640
[DEBUG] valid batch: (8, 3, 640, 640)
[DEBUG] lb batch   : (8, 3, 640, 640)


[fold 0] epoch 00 | loss 1.2955 | val_f1 0.8725 | lb_f1 0.8395


[fold 0] epoch 01 | loss 0.3267 | val_f1 0.8769 | lb_f1 0.8684


[fold 0] epoch 02 | loss 0.2373 | val_f1 0.8790 | lb_f1 0.8512


...

[fold 2] epoch 06 | loss 0.0310 | val_f1 0.9428 | lb_f1 0.9356


[fold 2] epoch 07 | loss 0.0118 | val_f1 0.9494 | lb_f1 0.9366


[fold 2] epoch 08 | loss 0.0078 | val_f1 0.9447 | lb_f1 0.9468


[fold 2] epoch 09 | loss 0.0070 | val_f1 0.9444 | lb_f1 0.9468
[fold 2] best val_f1: 0.9494 | saved: /root/CVProject/Model/artifacts/ckpts/best_fold2_val.pth
[fold 2] best lb_f1 : 0.9468 | saved: /root/CVProject/Model/artifacts/ckpts/best_fold2_lb.pth


[fold 2] LB eval (best_lb ckpt) | no-TTA: 0.9468 | TTA: 0.9136 | delta: -0.0332
saved: /root/CVProject/Model/artifacts/oof/lb_probs_by_fold.npy
saved: /root/CVProject/Model/artifacts/oof/lb_true.npy


### Total OOF macro F1 Cal + Save

In [44]:
oof_pred = oof_probs.argmax(axis=1)

# dev only OOF score (lb_holdout 제외)
oof_f1 = macro_f1(oof_true[dev_idx], oof_pred[dev_idx])
print("OOF macro f1 (dev only):", oof_f1)

if lb_idx is not None:
    print("note: lb_holdout indices excluded from OOF scoring:", len(lb_idx))

print("OOF macro f1:", oof_f1)

# oof 저장 (나중에 분석/앙상블에 유용)
oof_path = Path(cfg["paths"]["oof_dir"]) / "oof_probs.npy"
np.save(oof_path, oof_probs)
print("saved:", oof_path)


OOF macro f1 (dev only): 0.9542756260510676
note: lb_holdout indices excluded from OOF scoring: 314
OOF macro f1: 0.9542756260510676
saved: /root/CVProject/Model/artifacts/oof/oof_probs.npy


### aggressive vs safe 

In [45]:
# ===== 0) load train df / basics =====
train_df = load_train_df()
num_classes = train_df["target"].nunique()

# ===== 1) folds & checkpoints =====
ckpt_dir_p = Path(ckpt_dir)
available_folds = []
for p in ckpt_dir_p.glob("best_fold*_lb.pth"):
    # best_fold{fold}_lb.pth
    stem = p.stem
    fold = int(stem.split("best_fold")[1].split("_lb")[0])
    available_folds.append(fold)

available_folds = sorted(available_folds)
print("available_folds:", available_folds)

assert len(available_folds) > 0, f"no best_fold*_lb.pth found in {ckpt_dir_p}"

# ===== 2) build FULL train loader (valid transforms, no augmentation) =====
full_ds = doc_dataset(train_df, train_dir, transforms=get_valid_transforms(cfg["img_size"]), is_test=False)
full_loader = DataLoader(
    full_ds,
    batch_size=cfg["batch_size"],
    shuffle=False,
    num_workers=cfg["num_workers"],
    pin_memory=True
)

# ===== 3) compute & save probs_by_fold for FULL train (only once) =====
save_probs_path = Path(cfg["paths"]["oof_dir"]) / "train_probs_by_fold.npy"
save_true_path  = Path(cfg["paths"]["oof_dir"]) / "train_true.npy"

# 이미 저장된 게 있으면 로드해서 재사용 (시간 절약)
if save_probs_path.exists() and save_true_path.exists():
    train_probs_by_fold = np.load(save_probs_path, allow_pickle=True).item()
    train_true = np.load(save_true_path)
    print("✅ loaded cached:", save_probs_path.name, save_true_path.name)
else:
    train_probs_by_fold = {}
    train_true = None

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for fold in available_folds:
        ckpt_path = ckpt_dir_p / f"best_fold{fold}_lb.pth"
        print("infer full-train with:", ckpt_path.name)

        model = doc_classifier(
            model_name=cfg["model_name"],
            num_classes=num_classes,
            pretrained=False,
            dropout=cfg["dropout"]
        ).to(device)

        state = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(state)

        y_true, y_pred, probs = valid_one_epoch(model, full_loader, use_amp=cfg["use_amp"])
        train_probs_by_fold[fold] = probs.astype(np.float32)

        if train_true is None:
            train_true = y_true.astype(np.int64)

    np.save(save_probs_path, train_probs_by_fold)
    np.save(save_true_path, train_true)
    print("✅ saved:", save_probs_path)
    print("✅ saved:", save_true_path)

print("train_true shape:", train_true.shape)
print("example probs shape:", train_probs_by_fold[available_folds[0]].shape)

# ===== 4) define weights: uniform / aggressive(best_w) / safe(clip) =====
# ⚠️ 여기에 너의 best_w 결과를 그대로 넣어줘 (infer에서 나온 값)
best_folds = [0, 1, 2, 3, 4]
best_weights = np.array([0.03216075, 0.02183045, 0.02823855, 0.63305136, 0.2847189], dtype=np.float32)

# safe: clip then renorm (쏠림 완화)
safe_weights = np.clip(best_weights, 0.05, 0.5)
safe_weights = safe_weights / safe_weights.sum()

# uniform
uniform_weights = np.ones(len(best_folds), dtype=np.float32) / len(best_folds)

print("✅ aggressive weights sum:", float(best_weights.sum()))
print("✅ safe weights:", safe_weights, "| sum:", float(safe_weights.sum()))
print("✅ uniform weights:", uniform_weights, "| sum:", float(uniform_weights.sum()))

# ===== 5) eval function on a split =====
def ensemble_probs(idxs, folds, weights):
    probs_stack = np.stack([train_probs_by_fold[f][idxs] for f in folds], axis=0)  # (F, n, C)
    probs_ens = np.tensordot(weights, probs_stack, axes=(0, 0))  # (n, C)
    return probs_ens

def eval_on_seed(seed, holdout_ratio=0.2):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=holdout_ratio, random_state=seed)
    _, ho_idx = next(sss.split(np.zeros(len(train_true)), train_true))  # holdout idx only

    y = train_true[ho_idx]

    p_uni = ensemble_probs(ho_idx, best_folds, uniform_weights)
    p_agg = ensemble_probs(ho_idx, best_folds, best_weights)
    p_safe = ensemble_probs(ho_idx, best_folds, safe_weights)

    s_uni = macro_f1(y, p_uni.argmax(axis=1))
    s_agg = macro_f1(y, p_agg.argmax(axis=1))
    s_safe = macro_f1(y, p_safe.argmax(axis=1))

    return s_uni, s_agg, s_safe

# ===== 6) run multi seeds =====
seeds = [42, 202, 777, 1337, 2026]  # 원하면 더 늘려도 됨
rows = []
for sd in seeds:
    s_uni, s_agg, s_safe = eval_on_seed(sd, holdout_ratio=cfg.get("lb_holdout_ratio", 0.2))
    rows.append((sd, s_uni, s_agg, s_safe, s_agg - s_uni, s_safe - s_uni, s_agg - s_safe))
    print(f"[seed {sd}] uniform={s_uni:.5f} | aggressive={s_agg:.5f} | safe={s_safe:.5f} | "
          f"agg-uni={s_agg - s_uni:+.5f} | safe-uni={s_safe - s_uni:+.5f} | agg-safe={s_agg - s_safe:+.5f}")

rows = np.array(rows, dtype=object)

# ===== 7) summary =====
agg_win = sum(r[2] > r[3] for r in rows)  # aggressive > safe count
safe_win = sum(r[3] > r[2] for r in rows)

print("\n==== Summary ====")
print("seeds:", seeds)
print("aggressive wins vs safe:", agg_win, "/", len(seeds))
print("safe wins vs aggressive:", safe_win, "/", len(seeds))

# 평균/표준편차
uni_scores  = np.array([r[1] for r in rows], dtype=float)
agg_scores  = np.array([r[2] for r in rows], dtype=float)
safe_scores = np.array([r[3] for r in rows], dtype=float)

print(f"uniform  mean±std: {uni_scores.mean():.5f} ± {uni_scores.std():.5f}")
print(f"aggr     mean±std: {agg_scores.mean():.5f} ± {agg_scores.std():.5f}")
print(f"safe     mean±std: {safe_scores.mean():.5f} ± {safe_scores.std():.5f}")

available_folds: [0, 1, 2, 3, 4]
✅ loaded cached: train_probs_by_fold.npy train_true.npy
train_true shape: (1570,)
example probs shape: (1570, 17)
✅ aggressive weights sum: 1.0
✅ safe weights: [0.05349202 0.05349202 0.05349202 0.5349202  0.3046038 ] | sum: 1.0000001192092896
✅ uniform weights: [0.2 0.2 0.2 0.2 0.2] | sum: 1.0
[seed 42] uniform=0.95992 | aggressive=0.96633 | safe=0.95999 | agg-uni=+0.00642 | safe-uni=+0.00007 | agg-safe=+0.00635
[seed 202] uniform=1.00000 | aggressive=0.98546 | safe=0.99129 | agg-uni=-0.01454 | safe-uni=-0.00871 | agg-safe=-0.00583
[seed 777] uniform=0.99065 | aggressive=0.98528 | safe=0.98477 | agg-uni=-0.00537 | safe-uni=-0.00588 | agg-safe=+0.00051
[seed 1337] uniform=0.98830 | aggressive=0.97099 | safe=0.97392 | agg-uni=-0.01731 | safe-uni=-0.01438 | agg-safe=-0.00293
[seed 2026] uniform=0.98038 | aggressive=0.96923 | safe=0.96576 | agg-uni=-0.01116 | safe-uni=-0.01462 | agg-safe=+0.00346

==== Summary ====
seeds: [42, 202, 777, 1337, 2026]
aggressi